# 第10章 工具变量法

内生性：解释变量与扰动项相关

解决内生性的主要方法之一为工具变量法。

## 10.3 工具变量法的例子

假设供给方程扰动项的因素可以分解为两部分，即可观测的气温z，与不可观测的其他因素：
$$
q_t^s=\gamma+\delta p_t+\eta z_t+v_t
$$
假定气温$z_t$是前定变量，与需求方程的扰动项不相关，即$Cov(z_t,u_t)=0$。由于气温$z_t$的变化使得供给函数$q_t^s$沿着需求函数$q_t^t$移动，故可以估计出需求函数$q_t^d$。在这种情况下，称$z_t$为工具变量(Instrumental Variable，简记IV)。

在回归方程中(此处为需求方程)，一个有效的工具变量应满足以下两个条件。
1. 相关性(relevance)：工具变量与内生解释变量相关，即$Cov(z_t,p_t)\neq 0$。
2. 外生性(exogeneity)：工具变量与扰动项不相关，即$Cov(z_t,u_t)=0$。

## 10.4 二阶段最小二乘法

工具变量法一般通过二阶段最小二乘法(Two Stage Least Square，简记2SLS或TSLS)来实现，即通过做两个回归来完成。
- 第一阶段回归：用内生解释变量对工具变量回归，即$p_t\rightarrow z_t$，得到拟合值$\hat p_t$。
- 第二阶段回归：用被解释变量对第一阶段回归的拟合值进行回归，即$q_t\rightarrow \hat p_t$。

2SLS的实质是把内生解释变量$p_t$分成两部分，即由工具变量$z_t$,所造成的外生部分($\hat p_t$)以及与扰动项相关的其余部分($p_t-\hat p_t$)；然后，把被解释变量$q_t$对$p_t$中的外生部分($\hat p_t$)进行回归，从而满足OLS对前定变量的要求而得到一致估计。

2SLS的Stata命令格式为：
```
ivregress 2sls y x1 x2 (x3=z1 z2),robust first
```
- “y”为被解释变量
- “x1 x2”为外生解释变量
- “x3”为内生解释变量
- 而“z1 z2”为方程外的工具变量
- 选择项“robust”表示使用异方差稳健的标准误(默认为普通标准误)
- 选择项“first”表示显示第一阶段的回归结果

## 10.5 弱工具变量

如果工具变量与内生解释变量仅微弱地相关，则工具变量法估计量$\hat\beta_{IV}$的方差将变得很大。这种工具变量称为弱工具变量(weak instruments)。

为了检验是否存在弱工具变量，可在第一阶段回归中，检验所有方程外的工具变量(不含外生解释变量)的系数是否联合为零。比如，假设原模型为
$$
y=\beta_0+\beta_1 x+\beta_2 w+\epsilon
$$
其中，$x$为内生变量，而$w$为外生解释变量。假设有两个有效工具变量$z_1,z_2?$，则第一阶段回归为
$$
x=\alpha_0+\alpha_1 z_1+\alpha_2 z_2+\alpha_3 w+u
$$
然后检验$H_0:\alpha_1=\alpha_2=0$，即工具变量$z_1,z_2$的系数联合为0。一个经验规则(rule of thumb)是，如果此检验的F统计量大于10，则可拒绝“存在弱工具变量”的原假设，不必担心弱工具变量问题。

在Stata中作完2SLS回归后，可使用以下命令检验弱工具变量：
```
estat firststage
```

如果发现存在弱工具变量，可能的解决方法包括：
1. 寻找更强的工具变量；
2. 使用对弱工具变量更不敏感的有限信息最大似然估计法(Limited Informaton
Maximum Likelihood Estimation，简记LIML)。

LIML的Stata命令为：
```
ivregress liml y x1 x2 (x3=z1 z2)
```

## 10.6 对工具变量外生性的过度识别检验

过度识别检验(overidentification test)的大前提(maintained hypothesis)是该模型至少是恰好识别的，即有效工具变量至少与内生解释变量一样多。在此大前提下，过度识别检验的原假设为“$H_0$：所有工具变量都是外生的”。如果拒绝该原假设，则认为至少某个变量不是外生的，与扰动项相关。

假设共有m个方程外的工具变量$\{z_1,\cdots,z_m\}$，则过度识别的原假设为：
$$
H_0:Cov(z_1,\epsilon)=0,\cdots,Cov(z_m,\epsilon)=0
$$

在Stata中作完2SLS估计后，可使用以下命令进行过度识别检验：
```
estat overid
```

## 10.7 对解释变量内生性的豪斯曼检验：究竟该用OLS还是IV？

豪斯曼检验(Hausman specification test)的原假设为“$H_0$:所有解释变量均为外生变量”。如果$H_0$成立，则OLS与工具变量法都一致，即在大样本下$\hat\beta_{IV}$与$\hat\beta_{OLS}$都收敛于真实的参数值$\beta$，因此($\beta_{IV}-\beta_{OLS}$)(称为“对比向量”，vector of contrast)依概率收敛于0。反之如果$H_0$不成立，则工具变量法一致而OLS不一致，故($\beta_{IV}-\beta_{OLS}$)不会收敛于0。

豪斯曼检验的Stata命令为：
```
reg y x1 x2
estimates store ols                (存储OLS的结果，记为ols)
ivregress 2sls y x1(x2=z1 z2)      (假设x2为内生变量，z1,z2为IV)
estimates store iv                 (存储2SLS的结果，记为iv)
hausman iv ols,constant sigmamore  (根据存储结果进行豪斯曼检验)
```
- 选择项“sigmamore”表示统一使用更有效率的估计量(即OLS)所对应的残差来计算扰动项方差$\hat\sigma^2$。
- 选择项“constant”表示$\beta_{IV}$与$\beta_{OLS}$中都包括常数项(默认不含常数项)。

传统豪斯曼检验不适用异方差的情形(OLS只在球形扰动项的情况下才最有效率)。而改进的杜宾-吴-豪斯曼检验(Durbin-Wu-Hausman Test,简记DWH)即使在异方差的情况下也适用。

在Stata中作完2SLS估计后，可输人以下命令进行异方差稳健的DWH检验：
```
estat endogenous
```

## 10.9 工具变量法的 Stata 命令及实例

下面以数据集grilic.dta为例演示工具变量法，继续对教育投资回报率的探讨。此数据集的主要变量包括：
- lnw(工资对数)
- s(教育年限)
- expr(工龄)
- tenure(在现单位的工作年数)
- iq(智商)
- med(母亲的教育年限)
- kww(在“knowledge of the World of Work”测试中的成绩)
- rns(美国南方虚拟变量，住在南方=1)
- smsa(大城市虚拟变量，住在大城市=1)

(1)作为参照系，首先进行 OLS 回归，并使用稳健标准误。

In [1]:
use data/grilic.dta,clear
* 我们主要感兴趣的关键解释变量为s(教育年限),而expr,tenure,rns,smsa为控制变量。
reg lnw s expr tenure rns smsa,r


Linear regression                               Number of obs     =        758
                                                F(5, 752)         =      84.05
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3521
                                                Root MSE          =     .34641

------------------------------------------------------------------------------
             |               Robust
         lnw | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
           s |    .102643   .0062099    16.53   0.000     .0904523    .1148338
        expr |   .0381189   .0066144     5.76   0.000      .025134    .0511038
      tenure |   .0356146   .0079988     4.45   0.000     .0199118    .0513173
         rns |  -.0840797    .029533    -2.85   0.005    -.1420566   -.0261029
        smsa |

结果显示，教育投资的年回报率高达10.26而且在1%水平上显著不为0。这意味着，多受一年教育，则未来工资将高出10.26。

这个教育投资回报率似乎太高了。可能的原因是，由于遗漏变量“能力”与教育年限正相关，故“能力”对工资的贡献也被纳入教育的贡献，因此高估了教育的回报率。

(2)引入智商(iq)作为“能力”的代理变量(proxy)，再进行OLS回归。

In [2]:
reg lnw s iq expr tenure rns smsa,r


Linear regression                               Number of obs     =        758
                                                F(6, 751)         =      71.89
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3600
                                                Root MSE          =     .34454

------------------------------------------------------------------------------
             |               Robust
         lnw | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
           s |   .0927874   .0069763    13.30   0.000     .0790921    .1064826
          iq |   .0032792   .0011321     2.90   0.004     .0010567    .0055016
        expr |   .0393443   .0066603     5.91   0.000     .0262692    .0524193
      tenure |    .034209   .0078957     4.33   0.000     .0187088    .0497092
         rns |

加入“能力”的代理变量iq后，教育投资回报率下降为9.28更为合理些，但仍然显得过高。

(3)由于用iq来度量能力存在“测量误差”，故iq是内生变量。为此，考虑使用变量(med，kww)作为iq的工具变量。显然，母亲的教育年限(med)与KWW测试成绩(kww)都与iq正相关，并假设med与kww为外生。

In [3]:
* 下面进行2SLS回归，使用稳健标准误，并显示第一阶段的回归结果。
ivregress 2sls lnw s expr tenure rns smsa (iq=med kww),r first


First-stage regressions
-----------------------

                                                       Number of obs =     758
                                                       F(7, 750)     =   47.74
                                                       Prob > F      =  0.0000
                                                       R-squared     =  0.3066
                                                       Adj R-squared =  0.3001
                                                       Root MSE      = 11.3931

------------------------------------------------------------------------------
             |               Robust
          iq | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
           s |   2.467021   .2327755    10.60   0.000     2.010052     2.92399
        expr |  -.4501353   .2391647    -1.88   0.060    -.9196471    .0193766
      tenure |   .2059531    .269562     0.7

上表显示，教育投资回报率降为6.08且在1%平上显著；比较合理。

(4)下面进行过度识别检验：

In [4]:
estat overid


  Test of overidentifying restrictions:

  Score chi2(1)          =  .151451  (p = 0.6972)


由于p值为0.697，故接受原假设，认为(med,kuow)外生，与扰动项不相关。

(5)进一步考察有效工具变量的另一条件，即工具变量与内生变量的相关性。从第一阶段的回归结果可以看出，工具变量(med,kww)对内生变量iq均有较好的解释力，p值都小于0.05。正式检验须计算第一阶段回归的普通(非稳健)F统计量，故首先使用普通标准误重新进行2SLS估计。

In [5]:
quietly ivregress 2sls lnw s expr tenure rns smsa (iq=med kww)	
estat firststage


  First-stage regression summary statistics
  --------------------------------------------------------------------------
               |            Adjusted      Partial
      Variable |   R-sq.       R-sq.        R-sq.      F(2,750)   Prob > F
  -------------+------------------------------------------------------------
            iq |  0.3066      0.3001       0.0382       14.9058    0.0000
  --------------------------------------------------------------------------


  Minimum eigenvalue statistic = 14.9058     

  Critical Values                      # of endogenous regressors:    1
  H0: Instruments are weak             # of excluded instruments:     2
  ---------------------------------------------------------------------
                                     |    5%     10%     20%     30%
  2SLS relative bias                 |         (not available)
  -----------------------------------+---------------------------------
                                     |   10%     15%    

由于检验第一阶段回归的两个工具变量系数联合显著性的F统计量为14.91，超过10,故认为不存在弱工具变量。

(6)为了稳健起见，下面使用对弱工具变量更不敏感的有限信息最大似然法(LIML):

In [6]:
ivregress liml lnw s expr tenure rns smsa (iq=med kww),r


Instrumental-variables LIML regression            Number of obs   =        758
                                                  Wald chi2(6)    =     369.62
                                                  Prob > chi2     =     0.0000
                                                  R-squared       =     0.2768
                                                  Root MSE        =     .36454

------------------------------------------------------------------------------
             |               Robust
         lnw | Coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
          iq |   .0139764   .0060681     2.30   0.021     .0020831    .0258697
           s |   .0606362    .019034     3.19   0.001     .0233303    .0979421
        expr |   .0433416   .0074185     5.84   0.000     .0288016    .0578816
      tenure |   .0296237    .008323     3.56   0.000     .0133109    .0459364
         rns |

LIML的系数估计值与2SLS非常接近，从侧面印证了“不存在弱工具变量”。

(7)使用工具变量法的前提是存在内生解释变量。为此须进行豪斯曼检验，其原假设为“所有解释变量均为外生”，即不存在内生变量。

In [7]:
* 由于传统的豪斯曼检验建立在同方差的前提下，故回归中未使用稳健标准误(没有用选择项“r”)
qui reg lnw iq s expr tenure rns smsa
estimates store ols
qui ivregress 2sls lnw s expr tenure rns smsa (iq=med kww)
estimates store iv
hausman iv ols,constant sigmamore


Note: the rank of the differenced variance matrix (1) does not equal the number
        of coefficients being tested (7); be sure this is what you expect, or
        there may be problems computing the test.  Examine the output of your
        estimators for anything unexpected and possibly consider scaling your
        variables so that the coefficients are on a similar scale.

                 ---- Coefficients ----
             |      (b)          (B)            (b-B)     sqrt(diag(V_b-V_B))
             |       iv          ols         Difference       Std. err.
-------------+----------------------------------------------------------------
          iq |    .0139284     .0032792        .0106493        .0054318
           s |    .0607803     .0927874        -.032007        .0163254
        expr |    .0433237     .0393443        .0039794        .0020297
      tenure |    .0296442      .034209       -.0045648        .0023283
         rns |   -.0435271    -.0745325        .0310054     

上表显示，p值(Prob>chi2)为0.0499，故可在5%显著性水平上拒绝“所有解释变量均为外生”的原假设，认为iq为内生变量。

由于传统的豪斯曼检验在异方差的情形下不成立，下面进行异方差稳健的DWH检验：

In [8]:
estat endogenous


  Tests of endogeneity
  H0: Variables are exogenous

  Durbin (score) chi2(1)          =  3.87962  (p = 0.0489)
  Wu-Hausman F(1,750)             =  3.85842  (p = 0.0499)


上表提供了一个F统计量与一个$\chi^2$统计量，二者在大样本下渐近等价。由于二者的p值都小于0.05，故认为iq为内生解释变量。

(8)汇报结果：如果希望将以上各种估计法的系数及标准误列在同一表格中，可使用以下命令：

In [9]:
qui reg lnw s expr tenure rns smsa,r
est sto ols_no_iq
qui reg lnw iq s expr tenure rns smsa,r
est sto ols_with_iq
qui ivregress 2sls lnw s expr tenure rns smsa (iq=med kww),r
est sto tsls
qui ivregress liml lnw s expr tenure rns smsa (iq=med kww),r
est sto liml
* 选择项“b”表示显示回归系数，而选择项“se”表示显示标准误。
estimates table ols_no_iq ols_with_iq tsls liml,b se 


------------------------------------------------------------------
    Variable | ols_no_iq    ols_with~q      tsls         liml     
-------------+----------------------------------------------------
           s |  .10264304    .09278735    .06078035    .06063623  
             |  .00620988    .00697626    .01895051    .01903397  
        expr |   .0381189    .03934425    .04332367    .04334159  
             |  .00661439    .00666033    .00741179     .0074185  
      tenure |  .03561456    .03420896    .02964421    .02962365  
             |  .00799884    .00789567    .00831697    .00832297  
         rns | -.08407974   -.07453249   -.04352713   -.04338751  
             |  .02953295    .02997719    .03447789    .03452902  
        smsa |  .13966664    .13673691    .12722244     .1271796  
             |  .02805598    .02777116    .02974144    .02975994  
          iq |               .00327916    .01392844    .01397639  
             |               .00113212    .00603931    .00606

In [10]:
* 如果希望用一颗星表示10%显著性水平，两颗星表示5%显著性水平，而三颗星表示1%显著性水平，可使用如下命令：
estimates table ols_no_iq ols_with_iq tsls liml,star(0.1 0.05 0.01)


------------------------------------------------------------------------------
    Variable |   ols_no_iq      ols_with_iq        tsls            liml       
-------------+----------------------------------------------------------------
           s |  .10264304***    .09278735***    .06078035***    .06063623***  
        expr |   .0381189***    .03934425***    .04332367***    .04334159***  
      tenure |  .03561456***    .03420896***    .02964421***    .02962365***  
         rns | -.08407974***   -.07453249**    -.04352713      -.04338751     
        smsa |  .13966664***    .13673691***    .12722244***     .1271796***  
          iq |                  .00327916***    .01392844**     .01397639**   
       _cons |   4.103675***    3.8951718***    3.2180433***    3.2149943***  
------------------------------------------------------------------------------
                                           Legend: * p<.1; ** p<.05; *** p<.01


Stata官方命令“estimates table”无法同时显示回归系数、标准误与表示显著性的星号。为此，下载非官方命令“estout”。
```
ssc install estout
```

In [11]:
* 选择项“se”表示在括弧中显示标准误(默认显示t统计量，如果使用选择项“p”则显示p值)
* 选择项“r2”表示显示R^2
* 选择项“mtitle”表示使用模型名称(model title)作为表中每列的标题(默认以被解释变量为标题)
* 选择项“star(*0.1**0.05***0.01)”表示以星号表示显著性水平
esttab ols_no_iq ols_with_iq tsls liml,se r2 mtitle star(* 0.1 ** 0.05 *** 0.01)


----------------------------------------------------------------------------
                      (1)             (2)             (3)             (4)   
                ols_no_iq     ols_with_iq            tsls            liml   
----------------------------------------------------------------------------
s                   0.103***       0.0928***       0.0608***       0.0606***
                (0.00621)       (0.00698)        (0.0190)        (0.0190)   

expr               0.0381***       0.0393***       0.0433***       0.0433***
                (0.00661)       (0.00666)       (0.00741)       (0.00742)   

tenure             0.0356***       0.0342***       0.0296***       0.0296***
                (0.00800)       (0.00790)       (0.00832)       (0.00832)   

rns               -0.0841***      -0.0745**       -0.0435         -0.0434   
                 (0.0295)        (0.0300)        (0.0345)        (0.0345)   

smsa                0.140***        0.137***        0.127***        0.1

In [12]:
* 如果要将上表输出到 Microsoft Word 文档，并以文件名iv来命名此文档，则可运行如下命令：
* esttab ols_no_iq ols_with_iq tsls liml using iv.rtf,se r2 mtitle star(* 0.1 ** 0.05 *** 0.01) replace